# Step 0: Preprocessing

Before we can match or classify anything, the raw data needs to be cleaned. The training file has noisy literals (HTML tags, double spaces, weird quotes, mixed casing, accents) and the test file has the same issues. Step 2 (semantic retrieval) will also need normalized versions of the official ICD descriptions.

This notebook reads the three raw files, cleans them, and saves three preprocessed CSVs that the next steps will use.

Files to upload to Colab before running:
- `codification_data.csv` (training data)
- `leaderboard_data.csv` (test data)
- `icd_d_p_pairs.csv` (official ICD reference)

Files produced at the end:
- `train_preprocessed.csv`
- `test_preprocessed.csv`
- `reference_preprocessed.csv`

## 1. Load the raw data

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import re

# Training data: pairs of (Code, Literal) used to learn from
train = pd.read_csv('codification_data.csv')

# Test data: literals only, we need to predict the Code
test = pd.read_csv('leaderboard_data.csv')

# Reference: official ICD-10 code dictionary with descriptions
reference = pd.read_csv('icd_d_p_pairs.csv')

print(f'Training:  {len(train):,} rows, columns: {list(train.columns)}')
print(f'Test:      {len(test):,} rows, columns: {list(test.columns)}')
print(f'Reference: {len(reference):,} rows, columns: {list(reference.columns)}')

Training:  13,700 rows, columns: ['Code', 'Literal']
Test:      6,667 rows, columns: ['id', 'Literal']
Reference: 179,742 rows, columns: ['Code', 'D_P', 'Description']


## 2. Inspect the data to understand what needs cleaning

In [2]:
# A random sample shows the variety of issues better than head()
train.sample(15, random_state=42)

,Code,Literal
10334,6110,absceso mama
10856,O9972,Dermatitis Atópica parto
1261,B020ZZZ,TAC cranial
8839,D72820,linfocitosis
7981,T83511A,ITU sonda vesical
11388,N9489,PELVIS CONGELADA
4879,O9832,Herpes genital parto
13359,34931,punción dural posparto
13236,D62,ANEMIA POST-QUIRÚRGICA
7231,F17210,Fumadora PART


In [3]:
# Look for specific problems: HTML tags, weird quotes, double spaces
# Note: we put the literal acute accent character directly in the regex (not as \u00b4)
# because the newer pandas/PyArrow regex engine doesn't accept that escape sequence.
problem_examples = train[
    train['Literal'].str.contains(r'<[^>]+>|  |[`´]', regex=True, na=False)
]
print(f'Rows with HTML or weird characters: {len(problem_examples)}')
problem_examples.head(10)

Rows with HTML or weird characters: 8


,Code,Literal
1768,64891,SGB 2
1844,8893,RM espinal
2942,08RK3JZ,FACO+`LIO OI
4583,V1252,historia tromboflebitis
7959,5533,Hernia d´hiatus
8983,F411,trastorn d´ansietat generalitzat
11871,0059,<font><font>Alimentaria intoxicación</font></f...
12670,3E0337Z,stp`


In [4]:
# Check for missing values
print('Missing values in each file:')
print(f'  Training:  {train.isnull().sum().to_dict()}')
print(f'  Test:      {test.isnull().sum().to_dict()}')
print(f'  Reference: {reference.isnull().sum().to_dict()}')

Missing values in each file:
  Training:  {'Code': 0, 'Literal': 0}
  Test:      {'id': 0, 'Literal': 0}
  Reference: {'Code': 0, 'D_P': 0, 'Description': 0}


## 3. Define the cleaning functions

We need two functions:

- `basic_clean`: fixes obvious problems but keeps text readable. The original `Literal` column stays untouched for reference; this becomes `Literal_clean`.
- `normalize`: aggressive cleaning for matching, lowercase and accents removed. This is what we will compare strings against in Steps 1 and 2.

In [5]:
def basic_clean(text):
    """
    Light cleaning that keeps the text readable.
    Removes HTML tags, fixes weird quote characters, collapses multiple spaces.
    """
    if pd.isna(text):
        return ''
    text = str(text)
    # Strip HTML tags like <font><font>...</font></font>
    text = re.sub(r'<[^>]+>', '', text)
    # Replace backticks and acute accents used as quotes with a normal apostrophe
    text = text.replace('`', "'").replace('´', "'")
    # Collapse runs of whitespace into a single space, trim ends
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalize(text):
    """
    Aggressive cleaning used for string matching in later steps.
    Builds on basic_clean, then lowercases and removes accents.
    For example: 'broncoespástica' becomes 'broncoespastica'.
    """
    text = basic_clean(text).lower()
    # Decompose accented characters into base + accent, then drop the accent
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    return text.strip()


# Quick sanity check on both functions
samples = [
    '<font><font>Alimentaria intoxicación</font></font>',
    'SGB  2',
    'Hernia d´hiatus',
    'Hiperreactividad Bronquial',
    'broncoespástica',
]
for s in samples:
    print(f'Original:   "{s}"')
    print(f'  basic:    "{basic_clean(s)}"')
    print(f'  normal:   "{normalize(s)}"')
    print()

Original:   "<font><font>Alimentaria intoxicación</font></font>"
  basic:    "Alimentaria intoxicación"
  normal:   "alimentaria intoxicacion"

Original:   "SGB  2"
  basic:    "SGB 2"
  normal:   "sgb 2"

Original:   "Hernia d´hiatus"
  basic:    "Hernia d'hiatus"
  normal:   "hernia d'hiatus"

Original:   "Hiperreactividad Bronquial"
  basic:    "Hiperreactividad Bronquial"
  normal:   "hiperreactividad bronquial"

Original:   "broncoespástica"
  basic:    "broncoespástica"
  normal:   "broncoespastica"



## 4. Apply cleaning to the training data

We keep the original `Literal` column untouched, and add two cleaned versions next to it.

In [6]:
train['Literal_clean'] = train['Literal'].apply(basic_clean)
train['Literal_norm']  = train['Literal'].apply(normalize)

# Count how many literals were actually modified by the basic clean
n_modified = (train['Literal'] != train['Literal_clean']).sum()
print(f'Training literals modified by basic_clean: {n_modified}')

if n_modified > 0:
    print('\nBefore vs after (basic_clean):')
    changed = train[train['Literal'] != train['Literal_clean']]
    for _, row in changed.iterrows():
        print(f'  "{row.Literal}" -> "{row.Literal_clean}"')

Training literals modified by basic_clean: 8

Before vs after (basic_clean):
  "SGB  2" -> "SGB 2"
  "RM  espinal" -> "RM espinal"
  "FACO+`LIO OI" -> "FACO+'LIO OI"
  "historia  tromboflebitis" -> "historia tromboflebitis"
  "Hernia d´hiatus" -> "Hernia d'hiatus"
  "trastorn d´ansietat generalitzat" -> "trastorn d'ansietat generalitzat"
  "<font><font>Alimentaria intoxicación</font></font>" -> "Alimentaria intoxicación"
  "stp`" -> "stp'"


## 5. Add useful derived columns

Three extra columns will help us in the next steps:

- `code_type`: 'diagnosis' if the code starts with a letter, 'procedure' if it starts with a digit. These are two different ICD coding systems and we may want to handle them separately.
- `chapter`: the first character of the code. This is the broadest ICD category, useful as a coarse classification target.
- `is_icd10`: True if the code exists in the official ICD-10 reference, False if it looks like a legacy ICD-9 code.

In [7]:
# code_type: letter first means diagnosis, digit first means procedure
train['code_type'] = train['Code'].astype(str).str[0].apply(
    lambda c: 'diagnosis' if c.isalpha() else 'procedure'
)

# chapter: just the first character
train['chapter'] = train['Code'].astype(str).str[0]

# is_icd10: check membership in the reference
ref_codes = set(reference['Code'].unique())
train['is_icd10'] = train['Code'].isin(ref_codes)

print('code_type distribution:')
print(train['code_type'].value_counts())
print()
print('is_icd10 distribution:')
print(train['is_icd10'].value_counts())
print()
print('chapter distribution (top 10):')
print(train['chapter'].value_counts().head(10))

code_type distribution:
code_type
diagnosis    8753
procedure    4947
Name: count, dtype: int64

is_icd10 distribution:
is_icd10
True     9943
False    3757
Name: count, dtype: int64

chapter distribution (top 10):
chapter
Z    1715
O    1505
0    1141
6     637
3     592
B     579
N     536
E     500
V     491
5     408
Name: count, dtype: int64


## 6. Attach the official ICD description to each training row

For ICD-10 codes we can pull in the official description from the reference. For ICD-9 codes the merge will leave Description empty. This is useful for Step 2 (semantic retrieval) and for sanity-checking predictions.

In [8]:
# Left-join: keep all training rows, fill in description where it exists
train = train.merge(
    reference[['Code', 'D_P', 'Description']],
    on='Code',
    how='left'
)

print(f'Training rows after merge: {len(train):,}')
print(f'Rows with a Description:    {train["Description"].notna().sum():,}')
print(f'Rows without (ICD-9):       {train["Description"].isna().sum():,}')
print()
print('Sample rows after merge:')
train[['Code', 'Literal_clean', 'D_P', 'Description', 'code_type']].sample(5, random_state=1)

Training rows after merge: 13,700
Rows with a Description:    9,943
Rows without (ICD-9):       3,757

Sample rows after merge:


,Code,Literal_clean,D_P,Description,code_type
4706,M1710,Gonastrosis,D,"Artrosis primaria unilateral, rodilla no espec...",diagnosis
6454,O9902,Anemia Ferropénica parto,D,Anemia que complica el parto,diagnosis
3709,P2889,bronquiolitis vrs rn,D,Otras afecciones respiratorias especificadas d...,diagnosis
10445,283,Adenoidectomía 2,NaN,NaN,procedure
10026,0W9G3ZZ,Paracentesis,P,"Drenaje en cavidad peritoneal, abordaje percut...",procedure


## 7. Apply the same cleaning to the test data

The test file only has `id` and `Literal`. We add the same `Literal_clean` and `Literal_norm` columns so we can match against training in Step 1.

In [9]:
test['Literal_clean'] = test['Literal'].apply(basic_clean)
test['Literal_norm']  = test['Literal'].apply(normalize)

n_modified_test = (test['Literal'] != test['Literal_clean']).sum()
print(f'Test literals modified by basic_clean: {n_modified_test}')

test.head(5)

Test literals modified by basic_clean: 2


,id,Literal,Literal_clean,Literal_norm
0,1,AMNIODRENAJE,AMNIODRENAJE,amniodrenaje
1,2,Hiperparatiroidismo primario,Hiperparatiroidismo primario,hiperparatiroidismo primario
2,3,MIGRANYA parto,MIGRANYA parto,migranya parto
3,4,VHC,VHC,vhc
4,5,Absceso mama izq,Absceso mama izq,absceso mama izq


## 8. Normalize the reference descriptions

Step 2 will compare test literals against the 180K official descriptions. The comparison needs to happen on the normalized form (lowercase, no accents) so this is the right place to pre-compute it.

In [10]:
reference['Description_norm'] = reference['Description'].apply(normalize)

reference.sample(5, random_state=2)

,Code,D_P,Description,Description_norm
41924,S42102D,D,Fractura de parte no especificada de escápula ...,fractura de parte no especificada de escapula ...
75962,T3461XS,D,Congelación con necrosis de tejidos de cadera ...,congelacion con necrosis de tejidos de cadera ...
61835,S82022M,D,Fractura longitudinal desplazada de rótula izq...,fractura longitudinal desplazada de rotula izq...
147144,0PHQ34Z,P,"Inserción en metacarpo, izquierdo, de disposit...","insercion en metacarpo, izquierdo, de disposit..."
73680,T23292A,D,Quemadura de segundo grado de localizaciones m...,quemadura de segundo grado de localizaciones m...


## 9. Summary

In [11]:
print('Training data ready:')
print(f'  {len(train):,} rows, {len(train.columns)} columns')
print(f'  Columns: {list(train.columns)}')
print(f'  Diagnosis: {(train.code_type == "diagnosis").sum():,}, '
      f'Procedure: {(train.code_type == "procedure").sum():,}')
print(f'  ICD-10:    {train.is_icd10.sum():,}, ICD-9: {(~train.is_icd10).sum():,}')
print(f'  Unique codes: {train.Code.nunique():,}')
print()
print('Test data ready:')
print(f'  {len(test):,} rows, {len(test.columns)} columns')
print(f'  Columns: {list(test.columns)}')
print()
print('Reference ready:')
print(f'  {len(reference):,} rows, {len(reference.columns)} columns')
print(f'  Columns: {list(reference.columns)}')

Training data ready:
  13,700 rows, 9 columns
  Columns: ['Code', 'Literal', 'Literal_clean', 'Literal_norm', 'code_type', 'chapter', 'is_icd10', 'D_P', 'Description']
  Diagnosis: 8,753, Procedure: 4,947
  ICD-10:    9,943, ICD-9: 3,757
  Unique codes: 4,059

Test data ready:
  6,667 rows, 4 columns
  Columns: ['id', 'Literal', 'Literal_clean', 'Literal_norm']

Reference ready:
  179,742 rows, 4 columns
  Columns: ['Code', 'D_P', 'Description', 'Description_norm']


## 10. Save the preprocessed files

These three files will be the input for Step 1. After running this cell, click the folder icon on the left sidebar in Colab, right-click each file, and download them so you can re-upload them when running the next notebook.

In [ ]:
train.to_csv('train_preprocessed.csv', index=False)
test.to_csv('test_preprocessed.csv', index=False)
reference.to_csv('reference_preprocessed.csv', index=False)

print('Saved:')
print('  train_preprocessed.csv')
print('  test_preprocessed.csv')
print('  reference_preprocessed.csv')

Saved:
  train_preprocessed.csv
  test_preprocessed.csv
  reference_preprocessed.csv


: 